# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

In [10]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

In [11]:
audit_source_df = pd.read_csv(DATA_PATH)

required_columns = {
    "patient_id",
    "age",
    "sexe",
    "departement",
    "service",
    "type_admission",
    "nb_comorbidites",
    "imc",
    "dms_jours",
    "sejour_prolonge",
}

missing_columns = required_columns - set(audit_source_df.columns)

if missing_columns:
    raise KeyError(
        f"Colonnes obligatoires absentes : {sorted(missing_columns)}"
    )

print("Dimensions :", audit_source_df.shape)
print("\nColonnes :")
print(audit_source_df.columns.tolist())

print("\nTypes :")
display(audit_source_df.dtypes.to_frame("type"))

print("\nValeurs manquantes :")
display(
    audit_source_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("valeurs_manquantes")
)

Dimensions : (10000, 10)

Colonnes :
['patient_id', 'age', 'sexe', 'departement', 'service', 'type_admission', 'nb_comorbidites', 'imc', 'dms_jours', 'sejour_prolonge']

Types :


,type
patient_id,object
age,int64
sexe,object
departement,int64
service,object
type_admission,object
nb_comorbidites,int64
imc,float64
dms_jours,float64
sejour_prolonge,int64



Valeurs manquantes :


,valeurs_manquantes
patient_id,0
age,0
sexe,0
departement,0
service,0
type_admission,0
nb_comorbidites,0
imc,0
dms_jours,0
sejour_prolonge,0


In [5]:
# Fonction de calcul du Disparate Impact (règle des 4/5)

def disparate_impact(df, sensitive_col, target_col, positive_value,
                      min_group_size=None):
    
    grp = df.groupby(sensitive_col, observed=True)[target_col]
    sizes = grp.size()
    selection_rate = grp.apply(lambda x: (x == positive_value).mean())

    n_exclus = 0
    if min_group_size is not None:
        keep = sizes[sizes >= min_group_size].index
        n_exclus = len(selection_rate) - len(keep)
        selection_rate = selection_rate.loc[keep]

    sr_min, sr_max = selection_rate.min(), selection_rate.max()
    di = sr_min / sr_max  # toujours ≤ 1 par construction (min/max)

    verdict = "⚠️ Signal" if di < 0.8 else "✅ OK"

    return {
        "Variable": sensitive_col,
        "N groupes": len(selection_rate),
        "N groupes exclus (< seuil)": n_exclus,
        "Selection Rate min": round(sr_min, 3),
        "Selection Rate max": round(sr_max, 3),
        "DI": round(di, 3),
        "Verdict": verdict
    }

In [9]:
# Critères de discrimination

sensitive_vars = ["sexe", "age", "departement","service","type_admission","nb_comorbidites"]

results = []
for var in sensitive_vars:
    results.append(
        disparate_impact(
            df=df, sensitive_col=var, target_col="sejour_prolonge",
            positive_value=0
        )
    )

di_table = pd.DataFrame(results).sort_values(by="DI").reset_index(drop=True)
di_table

,Variable,N groupes,N groupes exclus (< seuil),Selection Rate min,Selection Rate max,DI,Verdict
0,nb_comorbidites,8,0,0.200,0.776,0.258,⚠️ Signal
1,service,5,0,0.366,0.805,0.454,⚠️ Signal
2,age,77,0,0.376,0.799,0.471,⚠️ Signal
3,type_admission,2,0,0.480,0.671,0.715,⚠️ Signal
4,sexe,2,0,0.509,0.679,0.749,⚠️ Signal
5,departement,5,0,0.582,0.605,0.962,✅ OK


In [12]:
# Taux positif chez les femmes
taux_f_label = (
    audit_df.loc[
        audit_df["sexe"] == "F",
        "sejour_prolonge"
    ]
    .mean()
)

# Taux positif chez les hommes
taux_m_label = (
    audit_df.loc[
        audit_df["sexe"] == "M",
        "sejour_prolonge"
    ]
    .mean()
)

# Disparate Impact
di_label = taux_f_label / taux_m_label

print(f"Taux F : {taux_f_label:.3f}")
print(f"Taux M : {taux_m_label:.3f}")
print(f"DI étiquette : {di_label:.3f}")

Taux F : 0.321
Taux M : 0.491
DI étiquette : 0.653


In [13]:
# Taux positif prédit chez les femmes
taux_f_pred = (
    audit_df.loc[
        audit_df["sexe"] == "F",
        "y_pred"
    ]
    .mean()
)

# Taux positif prédit chez les hommes
taux_m_pred = (
    audit_df.loc[
        audit_df["sexe"] == "M",
        "y_pred"
    ]
    .mean()
)

# Disparate Impact
di_pred = taux_f_pred / taux_m_pred

print(f"Taux F : {taux_f_pred:.3f}")
print(f"Taux M : {taux_m_pred:.3f}")
print(f"DI prédiction : {di_pred:.3f}")


Taux F : 0.141
Taux M : 0.486
DI prédiction : 0.291


In [20]:
audit_df["groupe_age"] = np.where(
    audit_df["age"] >= 65,
    ">=65 ans",
    "<65 ans"
)


In [22]:
# Taux de sélection par groupe
selection_rates = (
    audit_df
    .groupby("groupe_age")["sejour_prolonge"]
    .mean()
)

taux_moins_65 = selection_rates["<65 ans"]
taux_plus_65 = selection_rates[">=65 ans"]

# Disparate Impact
di_age = taux_moins_65 / taux_plus_65

print(f"Taux de sélection <65 ans : {taux_moins_65:.3f}")
print(f"Taux de sélection >=65 ans : {taux_plus_65:.3f}")
print(f"DI âge : {di_age:.3f}")


Taux de sélection <65 ans : 0.328
Taux de sélection >=65 ans : 0.531
DI âge : 0.618


In [19]:
taux_plus_61_pred = (
    audit_df.loc[
        audit_df["groupe_age"] == ">40 ans",
        "y_pred"
    ]
    .mean()
)

taux_moins_61_pred = (
    audit_df.loc[
        audit_df["groupe_age"] == "<=40 ans",
        "y_pred"
    ]
    .mean()
)

di_age_pred = (
    taux_plus_61_pred /
    taux_moins_61_pred
)

print(
    "Taux >40 ans :",
    round(taux_plus_61_pred, 3)
)

print(
    "Taux <=40 ans :",
    round(taux_moins_61_pred, 3)
)

print(
    "DI âge (prédiction) :",
    round(di_age_pred, 3)
)

Taux >40 ans : 0.41
Taux <=40 ans : 0.09
DI âge (prédiction) : 4.558


**Variables sensibles identifiées**

**Variables sensibles directes**
- Sexe (sexe)
- Âge (age)
Ces variables sont susceptibles de créer des différences de traitement entre groupes de patients et justifient une analyse d'équité spécifique.

**Variables indirectes (proxys potentiels)**
- Département (departement): territoires différents
- Service (service): gériatrie → patients plus âgés
- Type d'admission (type_admission): accès aux soins
- Nombre de comorbidités (nb_comorbidites): corrélé à l'age

Ces variables peuvent être corrélées à certaines populations ou caractéristiques sensibles et doivent être prises en compte lors de l'interprétation des résultats.

**Variables non retenues pour l'analyse d'équité**
- patient_id (identifiant technique)
- imc
- dms_jours
- sejour_prolonge (variable cible)

## 2. Ressources (psutil)

In [33]:
import os
import time
import psutil
import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [34]:
df = pd.read_csv("../data/dms_dataset.csv")

X = df[["age", "nb_comorbidites", "imc"]].copy()

X["sexe_bin"] = (
    df["sexe"] == "M"
).astype(int)

y = df["sejour_prolonge"]

In [35]:
def measure_model(model, X, y, model_filename):
    """
    Mesure :
    - temps entraînement
    - temps inférence
    - mémoire RSS
    - taille du modèle
    """

    process = psutil.Process(os.getpid())

    memory_before = (
        process.memory_info().rss
        / 1024**2
    )

    start_train = time.perf_counter()

    model.fit(X, y)

    end_train = time.perf_counter()

    memory_after = (
        process.memory_info().rss
        / 1024**2
    )

    start_predict = time.perf_counter()

    _ = model.predict(X)

    end_predict = time.perf_counter()

    joblib.dump(model, model_filename)

    model_size_mb = (
        os.path.getsize(model_filename)
        / 1024**2
    )

    return {
        "temps_train_s":
            end_train - start_train,

        "temps_inference_s":
            end_predict - start_predict,

        "memoire_rss_mb":
            memory_after - memory_before,

        "taille_modele_mb":
            model_size_mb
    }

In [36]:
rf = RandomForestClassifier(
    n_estimators=60,
    max_depth=10,
    random_state=0
)

rf_results = measure_model(
    model=rf,
    X=X,
    y=y,
    model_filename="rf_model.joblib"
)

rf_results

{'temps_train_s': 0.5952765000110958,
 'temps_inference_s': 0.07132309998269193,
 'memoire_rss_mb': 3.58984375,
 'taille_modele_mb': 4.729775428771973}

In [40]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000
)

lr_results = measure_model(
    model=lr,
    X=X,
    y=y,
    model_filename="lr_model.joblib"
)

lr_results

{'temps_train_s': 0.07291359998635016,
 'temps_inference_s': 0.007141099980799481,
 'memoire_rss_mb': 0.9765625,
 'taille_modele_mb': 0.0011892318725585938}

In [41]:
comparison = pd.DataFrame([
    {
        "Modèle": "Random Forest",
        **rf_results
    },
    {
        "Modèle": "Logistic Regression",
        **lr_results
    }
])

comparison

,Modèle,temps_train_s,temps_inference_s,memoire_rss_mb,taille_modele_mb
0,Random Forest,0.5953,0.0713,3.5898,4.7298
1,Logistic Regression,0.0729,0.0071,0.9766,0.0012


## 3. Comparaison à 2 alternatives

In [ ]:
# TODO
